# Notebook 10 — SHAP / XAI Analysis

This notebook adds the explainability part of the experimental pipeline. It does not train new models and it does not change the final ranking from Notebook 09.

The purpose is to answer two questions:

1. Which water-quality parameters drive the predicted CCME class?
2. Which monthly lags are most influential for the next-month prediction?

The analysis uses two complementary SHAP approaches:

- **XGBoost + TreeExplainer:** exact SHAP values for tree models. This is the main explanation layer because XGBoost is one of the strongest global models in Notebook 09.
- **Deep learning + GradientExplainer:** approximate SHAP values for the best deep-learning representatives. This is used to check whether neural models rely on similar signals.

The main metric and model ranking remain in Notebook 09. This notebook only explains model behaviour.


**Reviewer update:** This version is aligned with the corrected N09 final comparison: SHAP is applied to selected representative classical and deep-learning models, not to every trained model. Federated models are evaluated in performance/comparison notebooks, while SHAP here is used for interpretability of the strongest non-federated representatives.


In [6]:
# 0.1 — Imports (install shap if missing)
import os, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
    import shap

import joblib
print(f"shap version: {shap.__version__}")

shap version: 0.51.0


In [7]:
# 0.2 — Constants and clean output folder
ACTIVE_CLIENTS = ["England", "Ireland", "USA"]
FEATURE_COLS = ["NH3","BOD","DO","PO4","pH","Temp","Nitrogen","Nitrate"]
N_FEATS, N_CLASSES, WINDOW = 8, 5, 12
CLASS_NAMES = ["Excellent","Good","Fair","Marginal","Poor"]
SEED = 42
np.random.seed(SEED)

# Sampling sizes for SHAP (keeps runtime manageable)
N_EXPLAIN = 2000
N_BACKGROUND = 200

import shutil
OUTPUT_DIR = Path("/kaggle/working/shap_analysis")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
TAB_DIR = OUTPUT_DIR / "tables";  TAB_DIR.mkdir(exist_ok=True)
print(f"Output: {OUTPUT_DIR}")


Output: /kaggle/working/shap_analysis


In [8]:
# 0.3 — Auto-discover paths
def find_file(marker):
    for root, dirs, files in os.walk("/kaggle/input"):
        if marker in files:
            return Path(root) / marker
    return None

def find_dir(marker):
    p = find_file(marker)
    return p.parent if p else None

N02_DIR = find_dir("preprocess_manifest.json")
TENSOR_DIR = N02_DIR / "tensors" if N02_DIR else None
ML_MODEL_DIR = find_dir("local_xgboost_england.joblib")
GRU_MODEL_DIR = find_dir("local_gru_england.pt") or find_dir("local_gru_usa.pt")
TX_MODEL_DIR  = find_dir("local_transformer_england.pt")

print(f"N02 tensors : {TENSOR_DIR}")
print(f"ML models   : {ML_MODEL_DIR}")
print(f"GRU models  : {GRU_MODEL_DIR}")
print(f"TX models   : {TX_MODEL_DIR}")

assert TENSOR_DIR and TENSOR_DIR.exists(), "N02 tensors not found."
assert ML_MODEL_DIR and ML_MODEL_DIR.exists(), "N08 classical ML models not found."
assert GRU_MODEL_DIR and GRU_MODEL_DIR.exists(), "N03 GRU models not found."
assert TX_MODEL_DIR and TX_MODEL_DIR.exists(), "N05 Transformer models not found."

print("\nPaths resolved.")


N02 tensors : /kaggle/input/datasets/hebacheblal/preprocess-outputs/tensors
ML models   : /kaggle/input/datasets/hebacheblal/classical-ml-baselines/models
GRU models  : /kaggle/input/datasets/hebacheblal/gru-baselines/models
TX models   : /kaggle/input/datasets/hebacheblal/transformer-baselines/models

Paths resolved.


In [9]:
# 0.4 — Load tensors (sequence + flattened) and build feature names
data = {}
for client in ACTIVE_CLIENTS:
    arr = np.load(TENSOR_DIR / f"{client.lower()}_stage1_sequences.npz")
    Xtr, ytr = arr["X_train"], arr["y_train"]
    Xte, yte = arr["X_test"],  arr["y_test"]
    data[client] = {
        "X_train_seq": Xtr, "y_train": ytr,
        "X_test_seq":  Xte, "y_test":  yte,
        "X_train_flat": Xtr.reshape(Xtr.shape[0], WINDOW*N_FEATS),
        "X_test_flat":  Xte.reshape(Xte.shape[0], WINDOW*N_FEATS),
    }
    arr.close()
    print(f"{client:<8}: train={Xtr.shape[0]:,} test={Xte.shape[0]:,}")

# Flattened feature names: index = step*8 + feat, step 0 = t-12 (oldest)
flat_names = []
for step in range(WINDOW):
    lag = WINDOW - step
    for f in FEATURE_COLS:
        flat_names.append(f"t-{lag}_{f}")
assert len(flat_names) == WINDOW * N_FEATS
print(f"\nFeature names built: {flat_names[0]} ... {flat_names[-1]}")

England : train=76,530 test=11,942
Ireland : train=27,778 test=5,887
USA     : train=30,222 test=1,979

Feature names built: t-12_NH3 ... t-1_Nitrate


In [10]:
# 0.5 — Aggregation helpers: 96 flat features -> 8 params and 12 lags
# Flat index convention: idx = step*8 + feat_idx ; step 0 = t-12, step 11 = t-1
def agg_flat_to_param(importance_96):
    """Sum importance over the 12 lags for each of the 8 parameters."""
    m = np.asarray(importance_96).reshape(WINDOW, N_FEATS)  # (step, feat)
    return m.sum(axis=0)  # (8,)

def agg_flat_to_lag(importance_96):
    """Sum importance over the 8 parameters for each of the 12 steps.
       Returns lags ordered t-12 ... t-1 (step order)."""
    m = np.asarray(importance_96).reshape(WINDOW, N_FEATS)
    return m.sum(axis=1)  # (12,)  index 0 = t-12

def agg_seq_to_param(importance_12x8):
    """DL SHAP (12,8) -> 8 params (sum over steps)."""
    return np.asarray(importance_12x8).sum(axis=0)

def agg_seq_to_lag(importance_12x8):
    """DL SHAP (12,8) -> 12 lags (sum over params)."""
    return np.asarray(importance_12x8).sum(axis=1)

LAG_LABELS = [f"t-{WINDOW-s}" for s in range(WINDOW)]  # t-12 ... t-1
print("Aggregation helpers ready.")
print("Lag order:", LAG_LABELS)

Aggregation helpers ready.
Lag order: ['t-12', 't-11', 't-10', 't-9', 't-8', 't-7', 't-6', 't-5', 't-4', 't-3', 't-2', 't-1']


## Part A — XGBoost SHAP

This section explains the XGBoost models using TreeExplainer. TreeExplainer gives exact Shapley values for tree ensembles, so this part is the most reliable XAI component.

To stay aligned with Notebook 09, the explained XGBoost model is selected as follows:

- England: Local XGBoost, because it is the best overall model for England.
- Ireland: Centralized XGBoost, because it is the best overall model for Ireland.
- USA: Local XGBoost as a strong classical reference, while the final best model for USA is Local GRU and is explained again in the deep-learning section.

This gives both final-model interpretability and cross-model comparison.


In [11]:
# A.1 — Load selected XGBoost models and compute SHAP
def normalize_shap(sv):
    """Return SHAP as (n_samples, n_features, n_classes)."""
    if isinstance(sv, list):                       # old API: list per class
        return np.stack(sv, axis=-1)
    sv = np.asarray(sv)
    if sv.ndim == 2:                               # binary / single output
        return sv[..., None]
    return sv                                      # already (n, f, c)

XGB_TARGETS = {
    "England": ("Local XGBoost", ML_MODEL_DIR / "local_xgboost_england.joblib"),
    "Ireland": ("Centralized XGBoost", ML_MODEL_DIR / "centralized_xgboost.joblib"),
    "USA": ("Local XGBoost", ML_MODEL_DIR / "local_xgboost_usa.joblib"),
}
for client, (_, path) in XGB_TARGETS.items():
    assert path.exists(), f"Missing XGBoost model for {client}: {path}"

xgb_model_labels = {}
xgb_param_imp = {}         # client -> (8,)
xgb_lag_imp   = {}         # client -> (12,)
xgb_perclass_param = {}    # client -> (8, n_classes)
xgb_shap_store = {}        # client -> (sv_array, X_sample)

for client in ACTIVE_CLIENTS:
    model_label, model_path = XGB_TARGETS[client]
    xgb_model_labels[client] = model_label
    model = joblib.load(model_path)

    Xte = data[client]["X_test_flat"]
    n = min(N_EXPLAIN, Xte.shape[0])
    idx = np.random.RandomState(SEED).choice(Xte.shape[0], n, replace=False)
    Xs = Xte[idx]

    explainer = shap.TreeExplainer(model)
    sv = normalize_shap(explainer.shap_values(Xs))   # (n, 96, n_classes)
    xgb_shap_store[client] = (sv, Xs)

    # Global importance per flat feature: mean|SHAP| over samples and classes
    imp96 = np.abs(sv).mean(axis=(0, 2))             # (96,)
    xgb_param_imp[client] = agg_flat_to_param(imp96)
    xgb_lag_imp[client]   = agg_flat_to_lag(imp96)

    # Per-class param importance: mean|SHAP| over samples, then aggregate
    perclass = np.abs(sv).mean(axis=0)               # (96, n_classes)
    xgb_perclass_param[client] = np.stack(
        [agg_flat_to_param(perclass[:, c]) for c in range(perclass.shape[1])], axis=1)

    print(f"{client:<8} ({model_label}): SHAP shape {sv.shape}  top param = "
          f"{FEATURE_COLS[int(np.argmax(xgb_param_imp[client]))]}")

print("\nXGBoost SHAP computed for all selected clients.")


England  (Local XGBoost): SHAP shape (2000, 96, 5)  top param = PO4
Ireland  (Centralized XGBoost): SHAP shape (2000, 96, 5)  top param = PO4
USA      (Local XGBoost): SHAP shape (1979, 96, 4)  top param = PO4

XGBoost SHAP computed for all selected clients.


In [12]:
# A.2 — Feature (parameter) importance figure — XGBoost
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, client in zip(axes, ACTIVE_CLIENTS):
    imp = xgb_param_imp[client]
    order = np.argsort(imp)
    ax.barh(np.array(FEATURE_COLS)[order], imp[order], color="#2E8B57")
    ax.set_title(f"{client} — {xgb_model_labels[client]}", fontsize=10)
    ax.set_xlabel("Mean |SHAP| (summed over lags)")
fig.suptitle("Water-quality parameter importance (XGBoost, TreeExplainer)",
             fontsize=12, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIG_DIR / "xgb_parameter_importance.png", dpi=160, bbox_inches="tight")
plt.show()

# Save table
param_tbl = pd.DataFrame({c: xgb_param_imp[c] for c in ACTIVE_CLIENTS}, index=FEATURE_COLS)
param_tbl.round(5).to_csv(TAB_DIR / "xgb_parameter_importance.csv")
print("Saved parameter importance.")
display(param_tbl.round(4))

Saved parameter importance.


,England,Ireland,USA
NH3,0.6011,0.4291,0.1945
BOD,0.2456,0.2610,0.0709
DO,0.0624,0.2261,0.7719
PO4,1.9139,1.4770,0.9538
pH,0.1062,0.1482,0.6907
Temp,0.1544,0.1305,0.7212
Nitrogen,0.1256,0.2406,0.4361
Nitrate,0.0543,0.1109,0.4077


In [13]:
# A.3 — Temporal (lag) importance figure — XGBoost
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=False)
for ax, client in zip(axes, ACTIVE_CLIENTS):
    imp = xgb_lag_imp[client]   # index 0 = t-12 ... 11 = t-1
    ax.plot(range(WINDOW), imp, "o-", color="#1F3A5F")
    ax.set_xticks(range(WINDOW)); ax.set_xticklabels(LAG_LABELS, rotation=45, fontsize=7)
    ax.set_title(f"{client} — {xgb_model_labels[client]}", fontsize=10)
    ax.set_xlabel("Lag (months before target)"); ax.set_ylabel("Mean |SHAP|")
fig.suptitle("Temporal importance: which months matter (XGBoost)",
             fontsize=12, fontweight="bold", y=1.05)
plt.tight_layout()
plt.savefig(FIG_DIR / "xgb_temporal_importance.png", dpi=160, bbox_inches="tight")
plt.show()

lag_tbl = pd.DataFrame({c: xgb_lag_imp[c] for c in ACTIVE_CLIENTS}, index=LAG_LABELS)
lag_tbl.round(5).to_csv(TAB_DIR / "xgb_temporal_importance.csv")
print("Saved temporal importance.")
display(lag_tbl.round(4))

Saved temporal importance.


,England,Ireland,USA
t-12,0.3208,0.2727,0.3384
t-11,0.2871,0.2265,0.2698
t-10,0.3049,0.2275,0.3148
t-9,0.1920,0.2115,0.3402
t-8,0.1022,0.1699,0.2464
t-7,0.1316,0.1601,0.6133
t-6,0.1488,0.2036,0.4302
t-5,0.1237,0.1383,0.2725
t-4,0.2287,0.2083,0.2627
t-3,0.2853,0.2539,0.4194


In [14]:
# A.4 — Per-class parameter importance (England: all 5 classes present)
client = "England"
perclass = xgb_perclass_param[client]   # (8, n_classes)
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(N_FEATS); w = 0.16
for ci, cname in enumerate(CLASS_NAMES):
    if ci < perclass.shape[1]:
        ax.bar(x + (ci-2)*w, perclass[:, ci], w, label=cname)
ax.set_xticks(x); ax.set_xticklabels(FEATURE_COLS)
ax.set_ylabel("Mean |SHAP|"); ax.set_title(f"{client} — per-class parameter importance (XGBoost)",
                                            fontweight="bold")
ax.legend(fontsize=8, ncol=5, loc="upper right")
plt.tight_layout()
plt.savefig(FIG_DIR / "xgb_england_per_class_importance.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved per-class importance for England.")

Saved per-class importance for England.


In [15]:
# A.5 — SHAP beeswarm summary (England, automatically selected class)
client = "England"
sv, Xs = xgb_shap_store[client]
# Pick the class with the largest total attribution for a representative view
class_idx = int(np.abs(sv).mean(axis=(0,1)).argmax())
print(f"Beeswarm for class: {CLASS_NAMES[class_idx]}")
plt.figure()
shap.summary_plot(sv[:, :, class_idx], Xs, feature_names=flat_names,
                  max_display=15, show=False)
plt.title(f"{client} — SHAP beeswarm (class={CLASS_NAMES[class_idx]}, top 15 features)",
          fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "xgb_england_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved beeswarm.")

Beeswarm for class: Poor
Saved beeswarm.


## Part B — Deep-learning SHAP

This section explains the best deep-learning representatives using GradientExplainer. GradientExplainer is approximate, so it is used as a validation layer rather than as the only explanation.

The selected deep-learning models are:

- England: Local Transformer.
- Ireland: Centralized Transformer.
- USA: Local GRU.

These are the best deep-learning representatives from the current comparison outputs. The goal is to check whether neural models rely on similar parameters and lags as XGBoost.


In [16]:
# B.1 — Model class definitions (must match saved state dicts)
import torch
import torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class GRUClassifier(nn.Module):
    def __init__(self, input_size=N_FEATS, hidden_size=64, num_layers=1,
                 num_classes=N_CLASSES, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        _, h = self.gru(x)
        return self.classifier(self.dropout(h[-1]))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float()*(-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)[:, :pe[:,1::2].shape[1]]
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class TransformerClassifier(nn.Module):
    def __init__(self, input_size=N_FEATS, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, num_classes=N_CLASSES, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=WINDOW+1)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward, dropout=dropout,
            batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)
    def forward(self, x):
        x = self.pos_encoder(self.input_proj(x))
        x = self.encoder(x).mean(dim=1)
        return self.classifier(self.dropout(x))

print(f"Model classes defined. Device: {device}")

Model classes defined. Device: cuda


In [17]:
# B.2 — Load DL models and compute GradientExplainer SHAP
# cuDNN's fused RNN backward cannot run in eval mode, which GradientExplainer
# requires (dropout off). Disabling cuDNN falls back to the native RNN
# implementation. This only affects GRU/LSTM explanation, not training.
torch.backends.cudnn.enabled = False

DL_TARGETS = []
if TX_MODEL_DIR and (TX_MODEL_DIR / "local_transformer_england.pt").exists():
    DL_TARGETS.append(("England", "Local Transformer", TransformerClassifier,
                       TX_MODEL_DIR / "local_transformer_england.pt"))
if TX_MODEL_DIR and (TX_MODEL_DIR / "centralized_transformer.pt").exists():
    DL_TARGETS.append(("Ireland", "Centralized Transformer", TransformerClassifier,
                       TX_MODEL_DIR / "centralized_transformer.pt"))
if GRU_MODEL_DIR and (GRU_MODEL_DIR / "local_gru_usa.pt").exists():
    DL_TARGETS.append(("USA", "Local GRU", GRUClassifier,
                       GRU_MODEL_DIR / "local_gru_usa.pt"))

assert DL_TARGETS, "No DL models found for GradientExplainer."

dl_model_labels = {}
dl_param_imp = {}
dl_lag_imp = {}

for client, mname, cls, path in DL_TARGETS:
    dl_model_labels[client] = mname
    model = cls().to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()

    Xtr = data[client]["X_train_seq"]
    Xte = data[client]["X_test_seq"]

    bg_idx = np.random.RandomState(SEED).choice(
        Xtr.shape[0], min(N_BACKGROUND, Xtr.shape[0]), replace=False
    )
    ex_idx = np.random.RandomState(SEED+1).choice(
        Xte.shape[0], min(N_EXPLAIN, Xte.shape[0]), replace=False
    )

    background = torch.tensor(Xtr[bg_idx], dtype=torch.float32, device=device)
    X_explain  = torch.tensor(Xte[ex_idx], dtype=torch.float32, device=device)

    explainer = shap.GradientExplainer(model, background)
    sv = explainer.shap_values(X_explain)

    if isinstance(sv, list):
        sv_arr = np.stack(sv, axis=-1)       # (n,12,8,c)
    else:
        sv_arr = np.asarray(sv)
        if sv_arr.ndim == 3:
            sv_arr = sv_arr[..., None]

    imp_12x8 = np.abs(sv_arr).mean(axis=(0, 3))   # (12,8)
    dl_param_imp[client] = agg_seq_to_param(imp_12x8)
    dl_lag_imp[client]   = agg_seq_to_lag(imp_12x8)

    print(f"{client:<8} ({mname}): SHAP {sv_arr.shape}  top param = "
          f"{FEATURE_COLS[int(np.argmax(dl_param_imp[client]))]}")

print("\nDL SHAP computed.")


England  (Local Transformer): SHAP (2000, 12, 8, 5)  top param = PO4
Ireland  (Centralized Transformer): SHAP (2000, 12, 8, 5)  top param = DO
USA      (Local GRU): SHAP (1979, 12, 8, 5)  top param = PO4

DL SHAP computed.


In [18]:
# B.3 — DL parameter & temporal importance figures and tables
if dl_param_imp:
    n = len(dl_param_imp)

    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False)
    for ax, client in zip(axes[0], dl_param_imp):
        imp = dl_param_imp[client]
        order = np.argsort(imp)
        ax.barh(np.array(FEATURE_COLS)[order], imp[order], color="#1F3A5F")
        ax.set_title(f"{client} — {dl_model_labels[client]}", fontsize=10)
        ax.set_xlabel("Mean |SHAP| (summed over lags)")
    fig.suptitle("Parameter importance (deep learning, GradientExplainer)",
                 fontsize=12, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "dl_parameter_importance.png", dpi=160, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(1, n, figsize=(5*n, 3.4), squeeze=False)
    for ax, client in zip(axes[0], dl_lag_imp):
        ax.plot(range(WINDOW), dl_lag_imp[client], "o-", color="#C0392B")
        ax.set_xticks(range(WINDOW))
        ax.set_xticklabels(LAG_LABELS, rotation=45, fontsize=7)
        ax.set_title(f"{client} — {dl_model_labels[client]}", fontsize=10)
        ax.set_xlabel("Lag")
        ax.set_ylabel("Mean |SHAP|")
    fig.suptitle("Temporal importance (deep learning)", fontsize=12, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "dl_temporal_importance.png", dpi=160, bbox_inches="tight")
    plt.show()

    dl_param_tbl = pd.DataFrame({c: dl_param_imp[c] for c in dl_param_imp}, index=FEATURE_COLS)
    dl_lag_tbl = pd.DataFrame({c: dl_lag_imp[c] for c in dl_lag_imp}, index=LAG_LABELS)

    dl_param_tbl.round(5).to_csv(TAB_DIR / "dl_parameter_importance.csv")
    dl_lag_tbl.round(5).to_csv(TAB_DIR / "dl_temporal_importance.csv")

    print("Saved DL importance figures and tables.")
    display(dl_param_tbl.round(4))
    display(dl_lag_tbl.round(4))


Saved DL importance figures and tables.


,England,Ireland,USA
NH3,0.4818,0.2373,0.0407
BOD,0.1885,0.3588,0.0248
DO,0.1415,0.5618,0.1346
PO4,1.6230,0.5198,0.4354
pH,0.1169,0.2571,0.1417
Temp,0.2794,0.3828,0.1387
Nitrogen,0.2875,0.4052,0.0958
Nitrate,0.2160,0.4666,0.1067


,England,Ireland,USA
t-12,0.3387,0.2804,0.0850
t-11,0.2766,0.2656,0.0934
t-10,0.2180,0.2415,0.0858
t-9,0.1780,0.2215,0.0870
t-8,0.1690,0.2285,0.0942
t-7,0.1620,0.2486,0.0796
t-6,0.1503,0.2563,0.0789
t-5,0.1652,0.2757,0.0849
t-4,0.1618,0.2531,0.0906
t-3,0.2257,0.2398,0.1044


## Part C — Cross-model validation and synthesis

This section compares the parameter importance obtained from XGBoost and deep learning. Agreement between the two model families is useful because they have different inductive biases.

If both model families highlight the same variables, the explanation is more likely to reflect a real data signal rather than a model-specific artifact.


In [19]:
# C.1 — Cross-model parameter importance comparison (normalised)
def normalize(v):
    v = np.asarray(v, dtype=float)
    s = v.sum()
    return v / s if s > 0 else v

rows = []
for client in dl_param_imp:   # clients with both XGB and DL explanations
    xgb_n = normalize(xgb_param_imp[client])
    dl_n  = normalize(dl_param_imp[client])
    for i, f in enumerate(FEATURE_COLS):
        rows.append({
            "Client": client,
            "Parameter": f,
            "XGBoost_importance": round(float(xgb_n[i]), 4),
            "DL_importance": round(float(dl_n[i]), 4),
        })

cross_df = pd.DataFrame(rows)
cross_df.to_csv(TAB_DIR / "cross_model_parameter_importance.csv", index=False)

# Rank correlation per client: saved because it is used in thesis interpretation
from scipy.stats import spearmanr
agreement_rows = []
cross_model_agreement = {}

print("Cross-model agreement (Spearman rank correlation of parameter importance):")
for client in dl_param_imp:
    rho, p_value = spearmanr(xgb_param_imp[client], dl_param_imp[client])
    rho = float(rho) if np.isfinite(rho) else np.nan
    p_value = float(p_value) if np.isfinite(p_value) else np.nan
    cross_model_agreement[client] = rho
    agreement_rows.append({
        "Client": client,
        "XGB_model": xgb_model_labels.get(client, ""),
        "DL_model": dl_model_labels.get(client, ""),
        "Spearman_rho": rho,
        "Spearman_p_value": p_value,
    })
    print(f"  {client}: rho = {rho:+.3f}")

agreement_df = pd.DataFrame(agreement_rows)
agreement_df.to_csv(TAB_DIR / "cross_model_agreement_spearman.csv", index=False)

# Side-by-side bar figure
if dl_param_imp:
    n = len(dl_param_imp)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 4), squeeze=False)
    for ax, client in zip(axes[0], dl_param_imp):
        x = np.arange(N_FEATS)
        w = 0.4
        ax.bar(x - w/2, normalize(xgb_param_imp[client]), w, label="XGBoost", color="#2E8B57")
        ax.bar(x + w/2, normalize(dl_param_imp[client]), w, label="Deep learning", color="#1F3A5F")
        ax.set_xticks(x)
        ax.set_xticklabels(FEATURE_COLS, rotation=45, fontsize=8)
        ax.set_title(f"{client}", fontsize=10)
        ax.set_ylabel("Normalised importance")
        ax.legend(fontsize=8)
    fig.suptitle("Cross-model parameter importance: XGBoost vs Deep Learning",
                 fontsize=12, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "cross_model_comparison.png", dpi=160, bbox_inches="tight")
    plt.show()

display(cross_df.head(16))
display(agreement_df.round(4))


Cross-model agreement (Spearman rank correlation of parameter importance):
  England: rho = +0.690
  Ireland: rho = -0.095
  USA: rho = +0.881


,Client,Parameter,XGBoost_importance,DL_importance
0,England,NH3,0.1842,0.1445
1,England,BOD,0.0753,0.0565
2,England,DO,0.0191,0.0424
3,England,PO4,0.5865,0.4867
4,England,pH,0.0325,0.0351
5,England,Temp,0.0473,0.0838
6,England,Nitrogen,0.0385,0.0862
7,England,Nitrate,0.0166,0.0648
8,Ireland,NH3,0.1419,0.0744
9,Ireland,BOD,0.0863,0.1125


,Client,XGB_model,DL_model,Spearman_rho,Spearman_p_value
0,England,Local XGBoost,Local Transformer,0.6905,0.0580
1,Ireland,Centralized XGBoost,Centralized Transformer,-0.0952,0.8225
2,USA,Local XGBoost,Local GRU,0.8810,0.0039


In [20]:
# C.2 — Synthesis table
summary_rows = []
print("="*70)
print("SHAP / XAI — SYNTHESIS")
print("="*70)

for client in ACTIVE_CLIENTS:
    top3 = np.array(FEATURE_COLS)[np.argsort(xgb_param_imp[client])[::-1][:3]]
    top_lag = LAG_LABELS[int(np.argmax(xgb_lag_imp[client]))]
    row = {
        "Client": client,
        "XGB_model": xgb_model_labels[client],
        "XGB_top_1_parameter": top3[0],
        "XGB_top_2_parameter": top3[1],
        "XGB_top_3_parameter": top3[2],
        "XGB_most_important_lag": top_lag,
    }

    if client in dl_param_imp:
        dl_top3 = np.array(FEATURE_COLS)[np.argsort(dl_param_imp[client])[::-1][:3]]
        dl_top_lag = LAG_LABELS[int(np.argmax(dl_lag_imp[client]))]
        rho = cross_model_agreement.get(client, np.nan)
        row.update({
            "DL_model": dl_model_labels[client],
            "DL_top_1_parameter": dl_top3[0],
            "DL_top_2_parameter": dl_top3[1],
            "DL_top_3_parameter": dl_top3[2],
            "DL_most_important_lag": dl_top_lag,
            "Top_parameter_agreement": bool(top3[0] == dl_top3[0]),
            "Cross_model_spearman_rho": rho,
        })
    summary_rows.append(row)

    print(f"\n{client}:")
    print(f"   XGBoost model: {xgb_model_labels[client]}")
    print(f"   XGBoost top-3 parameters: {', '.join(top3)}")
    print(f"   XGBoost most important lag: {top_lag}")
    if client in dl_param_imp:
        print(f"   DL model: {dl_model_labels[client]}")
        print(f"   DL top-3 parameters: {', '.join(dl_top3)}")
        print(f"   DL most important lag: {dl_top_lag}")
        print(f"   Cross-model Spearman rho: {rho:+.3f}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TAB_DIR / "shap_synthesis_summary.csv", index=False)
print("\nSaved synthesis table.")
display(summary_df)


SHAP / XAI — SYNTHESIS

England:
   XGBoost model: Local XGBoost
   XGBoost top-3 parameters: PO4, NH3, BOD
   XGBoost most important lag: t-1
   DL model: Local Transformer
   DL top-3 parameters: PO4, NH3, Nitrogen
   DL most important lag: t-1
   Cross-model Spearman rho: +0.690

Ireland:
   XGBoost model: Centralized XGBoost
   XGBoost top-3 parameters: PO4, NH3, BOD
   XGBoost most important lag: t-1
   DL model: Centralized Transformer
   DL top-3 parameters: DO, PO4, Nitrate
   DL most important lag: t-1
   Cross-model Spearman rho: -0.095

USA:
   XGBoost model: Local XGBoost
   XGBoost top-3 parameters: PO4, DO, Temp
   XGBoost most important lag: t-7
   DL model: Local GRU
   DL top-3 parameters: PO4, pH, Temp
   DL most important lag: t-1
   Cross-model Spearman rho: +0.881

Saved synthesis table.


,Client,XGB_model,XGB_top_1_parameter,XGB_top_2_parameter,XGB_top_3_parameter,XGB_most_important_lag,DL_model,DL_top_1_parameter,DL_top_2_parameter,DL_top_3_parameter,DL_most_important_lag,Top_parameter_agreement,Cross_model_spearman_rho
0,England,Local XGBoost,PO4,NH3,BOD,t-1,Local Transformer,PO4,NH3,Nitrogen,t-1,True,0.690476
1,Ireland,Centralized XGBoost,PO4,NH3,BOD,t-1,Centralized Transformer,DO,PO4,Nitrate,t-1,False,-0.095238
2,USA,Local XGBoost,PO4,DO,Temp,t-7,Local GRU,PO4,pH,Temp,t-1,True,0.880952


## Final reading

The SHAP analysis is used as supporting interpretability evidence. It should not be interpreted as a new performance experiment.

The expected thesis-safe reading is:

- Phosphate (`PO4`) is usually among the strongest predictors, especially for England.
- Recent months, especially `t-1`, are important for England and Ireland, which is consistent with one-month-ahead forecasting.
- USA may show less intuitive lag behaviour because it is the most affected by repeated/modal-value patterns and has only Excellent/Good classes in the test set.
- Agreement between XGBoost and deep-learning importance supports that the signal is not only an artifact of one model family.

The limitations must also be stated: SHAP explains the trained models, not the physical water system directly; GradientExplainer is approximate; and USA explanations must be interpreted cautiously because of modal-value dominance.


In [21]:
# C.3 — Zip outputs
import shutil
zip_path = shutil.make_archive(str(Path("/kaggle/working")/"shap_analysis"),
                               "zip", root_dir=OUTPUT_DIR)
print(f"Archive: {zip_path}")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file(): print(f"  {p.relative_to(OUTPUT_DIR)}")

Archive: /kaggle/working/shap_analysis.zip
  figures/cross_model_comparison.png
  figures/dl_parameter_importance.png
  figures/dl_temporal_importance.png
  figures/xgb_england_beeswarm.png
  figures/xgb_england_per_class_importance.png
  figures/xgb_parameter_importance.png
  figures/xgb_temporal_importance.png
  tables/cross_model_agreement_spearman.csv
  tables/cross_model_parameter_importance.csv
  tables/dl_parameter_importance.csv
  tables/dl_temporal_importance.csv
  tables/shap_synthesis_summary.csv
  tables/xgb_parameter_importance.csv
  tables/xgb_temporal_importance.csv
